# Decoding analysis after regressing out effect of contrast

We show how to do it in sensor space, but the same thing can be done in source space. 


In [ ]:
import mne
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import tqdm
import pickle as pkl

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# importing from our local utils
import sys
sys.path.append("../scripts/")
from params import fnames

## Loading and preparing data
Just so you have something to play around with, I've added some epochs to the `data_for_notebooks` folder here on uCloud.

In [ ]:
epochs_path = Path("/work/data_for_notebooks/0191_filtered_0.1-40_fir_ICA_meg-epo.fif")
epochs = mne.read_epochs(epochs_path)
epochs = epochs.drop_channels(epochs.info["bads"])
epochs.pick("meg")

# for the decoder we will combine pas scores 1 + 2 together, and 3 + 4 to have more trials
epochs.metadata["subjective_combined"] = np.where(
    epochs.metadata["subjective_response"].isin([1, 2]),
    0,
    1
)

In [ ]:
idx = np.where(
    epochs.metadata["epoch_type"] == "target"
)[0]

metadata = epochs.metadata.iloc[idx]
counts = metadata["subjective_combined"].value_counts()
print(counts)

# remove random trials from the subjective response with the largest number of trials
n = counts.min()

# Randomly select n trials from each response
metadata = metadata.groupby("subjective_combined").sample(
    n=n,
    random_state=99
)

# Get the original epoch indices
idx = metadata.index.to_numpy()

In [ ]:
epochs_subset = epochs[idx]
MEG_data = epochs_subset.get_data()
contrast = epochs_subset.metadata["target_contrast"].to_numpy()
pas_score = epochs_subset.metadata["subjective_combined"]

print("MEG data shape:", MEG_data.shape)
print("Contrast shape:", contrast.shape)

Now lets set up some useful scikit learn classes which we will be using throughout the notebook! 


In [ ]:
ncv=10
scaler = StandardScaler()
kfold = StratifiedKFold(ncv, shuffle=True)
clf = LogisticRegression(
    random_state = 99,
    C=1.0,
    max_iter=1000
)

# Regressing out effects of contrast
To avoid data leakage, we need to estimate the effect of contrast using only the training data and then use those regression coefficients to remove the contrast-related activity from both the training and test data.

However, first we'll ignore cross-validation and use all the data to get an idea of how the regression works.

Let's begin by only taking the data for one timepoint



In [ ]:
sample = 140
epochs.times[sample]

# For simplicity looking at one time point
MEG_data_t = MEG_data[:, :, sample]

print(MEG_data_t.shape)

Each row corresponds to one trial, and each column corresponds to one MEG sensor. We want to ask: **How much of the activity at each sensor can be explained by the contrast?**

We do this by fitting a linear regression model:

$$
Y = X\beta + \epsilon
$$

where $Y$ is our **MEG data matrix**, $X$ is the **design matrix** containing the contrast, $\beta$ contains the regression coefficients, and $\epsilon$ represents the unexplained variance.

Our MEG data matrix is:

$$
Y =
\begin{bmatrix}
y_{1,1} & y_{1,2} & \cdots & y_{1,M} \\
y_{2,1} & y_{2,2} & \cdots & y_{2,M} \\
\vdots & \vdots & \ddots & \vdots \\
y_{N,1} & y_{N,2} & \cdots & y_{N,M}
\end{bmatrix}
$$

where $N$ is the number of trials and $M$ is the number of MEG sensors.

We then create our design matrix:

$$
X =
\begin{bmatrix}
1 & c_1 \\
1 & c_2 \\
\vdots & \vdots \\
1 & c_N
\end{bmatrix}
$$

The first column contains the **intercept**, and the second column contains the **contrast $c$**.

In [ ]:
X_design = np.column_stack([
    np.ones(len(contrast)),
    contrast
])
print("Design matrix shape:", X_design.shape)

This gives us: [intercept, contrast] for every trial.


We can now estimate the regression coefficients **simultaneously for all sensors**. The resulting coefficients are:

$$
\beta =
\begin{bmatrix}
\beta_0^{(1)} & \beta_0^{(2)} & \cdots & \beta_0^{(M)} \\
\beta_1^{(1)} & \beta_1^{(2)} & \cdots & \beta_1^{(M)}
\end{bmatrix}
$$

where $\beta_1^{(M)}$ tells us how strongly the activity at sensor $M$ is related to the contrast.

We estimate these coefficients using least squares:

$$
\hat{\beta} = (X^\top X)^{-1}X^\top Y
$$


In [ ]:
beta = np.linalg.lstsq(
    X_design,
    MEG_data_t,
    rcond=None
)[0]
print("Beta shape:", beta.shape)

Next, we use these coefficients to predict the MEG activity that can be explained by contrast:

$$
\hat{Y} = X\hat{\beta}
$$

This gives us the **contrast-related component** of the MEG activity for each trial and sensor.


In [ ]:
MEG_predicted = X_design @ beta
print("Predicted MEG shape:", MEG_predicted.shape)


We can then remove this component from the original data to obtain the residual activity:

$$
Y_{\mathrm{residual}} = Y - \hat{Y}
$$

The residuals represent the MEG activity that is **not explained by the linear relationship with contrast**.

In [ ]:
MEG_residuals = MEG_data_t - MEG_predicted

print("Residual MEG shape:", MEG_residuals.shape)

Let's plot the evoked response before and after to compare!


In [ ]:
# make a copy so we don't overwrite the original data
epochs_subset_after = epochs_subset.copy()

# Rrplace the data at this time point with the residual data
# hacky: we are just using the Epochs object for convenient plotting
epochs_subset_after._data[:, :, sample] = MEG_residuals

# average across trials
evoked_before = epochs_subset.average()
evoked_after = epochs_subset_after.average()

fig, axes = plt.subplots(
    1, 2,
    dpi=300,
    gridspec_kw={"wspace": 0.3}
)

for cond, evoked, ax in zip(["before", "after"],[evoked_before, evoked_after], axes):
    mne.viz.plot_evoked_topomap(
        evoked,
        axes=ax,
        times=epochs.times[sample],
        show=False,
        colorbar=False,
    )
    ax.set_title(cond)


<div class="alert alert-block alert-info">
<b>Challenge:</b> Compare the "amplitudes" before and after regression! Why are they suddenly on very different scales? Try to plot them or look at the numerical values.

💡 <b>Hint:</b> Think about what happens to the <b>intercept</b> when we calculate the residuals.

</div>


Importantly, this is only a demonstration of the regression. If we subsequently use these residuals for decoding, we have a problem: the regression coefficients were estimated using all trials, including trials that will later belong to our test sets.

## Interfacing with scikit-learn so we can Pipeline!

Here we use a hacky workaround to make the regression step work within a scikit-learn `Pipeline`.

The problem is that our regression step needs access to both the MEG data (our `X`) and the contrast. However, the classifier should only receive the residualized MEG data—not the contrast itself.

To make this work, we temporarily add the contrast as an additional feature (sensor) dimension:

The `RegressOutContrast` transformer then uses this contrast column to estimate the regression coefficients. After this step, the contrast column is removed.


We also add a small custom transformer that prints the shape of `X` at different points in the pipeline. This allows us to verify that the contrast is indeed present initially and that it has been removed before the data reach the classifier.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class RegressOutContrast(BaseEstimator, TransformerMixin):

    def __init__(self, n_sensors):
        self.n_sensors = n_sensors

    def fit(self, X, y=None):

        # Separate MEG and contrast
        meg = X[:, :self.n_sensors]
        contrast = X[:, self.n_sensors]

        # Design matrix
        self.X_design_ = np.column_stack([
            np.ones(len(contrast)),
            contrast
        ])

        # Fit regression using TRAINING data only
        self.beta_ = np.linalg.lstsq(
            self.X_design_,
            meg,
            rcond=None
        )[0]

        return self

    def transform(self, X):

        # Separate MEG and contrast
        meg = X[:, :self.n_sensors]
        contrast = X[:, self.n_sensors]

        # Design matrix for these trials
        X_design = np.column_stack([
            np.ones(len(contrast)),
            contrast
        ])

        # Predicted MEG
        meg_hat = X_design @ self.beta_

        # Remove contrast-related activity
        meg_residuals = meg - meg_hat

        return meg_residuals


class PrintShape(BaseEstimator, TransformerMixin):

    def __init__(self, name=""):
        self.name = name
        self.print_indexes=[0]
        self.i = 0


    def fit(self, X, y=None):
        return self

    def transform(self, X):
        
        if self.i in self.print_indexes:
            print(f"{self.name}: X shape = {X.shape}")
        self.i += 1


        return X

Now, lets try and classify what subjective response was given!

In [ ]:
X_with_contrast = np.column_stack([
    MEG_data_t,
    contrast
])

print("Input to pipeline:", X_with_contrast.shape) # trials, predictors (n sensors + 1 )


In [ ]:
n_sensors = MEG_data.shape[1]

pipeline_regress_out = Pipeline([
    ("print_shape1", PrintShape(name="Before regression")),
    ("regress_out_contrast", RegressOutContrast(n_sensors=n_sensors)),
    ("print_shape2", PrintShape(name="After regression")),
    ("scaler", scaler),
    ("clf", clf)
])

scores_reg = cross_val_score(
    pipeline_regress_out,
    X_with_contrast,
    pas_score,
    cv=kfold
)

pipeline = Pipeline([
    ("scaler", scaler),
    ("clf", clf)
])


scores = cross_val_score(
    pipeline,
    X_with_contrast[:, :n_sensors],
    pas_score,
    cv=kfold
)


In [ ]:
print(f"Mean accuracy with contrast regressed out = {scores_reg.mean():.3f}")
print(f"Mean accuracy when contrast is not regressed out = {scores.mean():.3f}")

## Decoding across time points after removing effect of contrast

In [ ]:
scores = np.empty((ncv, MEG_data.shape[-1]))

pipeline_regress_out_no_print = Pipeline([
    ("regress_out_contrast", RegressOutContrast(n_sensors=n_sensors)),
    ("scaler", scaler),
    ("clf", clf)
])

for t in tqdm(range(MEG_data.shape[-1])): # loop over time points
    X_t = MEG_data[:, :, t]
    
    X_with_contrast = np.column_stack([
        X_t,
        contrast
    ])
    scores_t = cross_val_score(
        pipeline_regress_out_no_print, 
        X_with_contrast, 
        pas_score, 
        cv=kfold
        )
        
    scores[:, t] = scores_t

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=300)

# Plot accuracy of each cross-validation fold
for score_cv in scores:
    ax.plot(
        epochs.times,
        score_cv*100,
        alpha=0.05,
        linewidth=1,
        color="darkgreen"
    )

# Plot the average across cross-validation folds
average_score = scores.mean(axis=0)

ax.plot(
    epochs.times,
    average_score*100,
    linewidth=2,
    color="darkgreen",
    label="Decoding accuracy\nacross CV folds"
)

# Aesthetics
ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Time (s)")
ax.axvline(0, color = "grey", linestyle = "--", label="Stimulus presentation")


# remove top and right spines
for spine_label in ["top", "right"]:
    ax.spines[spine_label].set_visible(False)

# add legend
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

# Challenge: Do it in source space

Note that the source time courses have been morphed to fsaverage!

In [ ]:
stcs_path = Path("/work/data_for_notebooks/0195_morphed_to_fsaverage_filtered_0.1-40_fir_ICA_MNE.pkl")
with open(stcs_path, "rb") as f:
    stcs = pkl.load(f)